# Fase 2 — Ingestão Incremental em Batch dos Logs da Urna

Visão geral da ingestão incremental em batch dos logs da urna eletrônica (`fact_urna_logs` — 15 colunas 1:1 com `message_text` mascarado para números de 5+ dígitos como `<ID_REDACTED>` e sem `mac_code`) por UF e Turno em Parquet ZSTD particionado (`election_year=2026/turn=T/uf=UF`).

In [ ]:
from pathlib import Path
import polars as pl
import pyarrow as pa

# Parâmetros idempotentes de execução
ELECTION_YEAR = 2026
TARGET_TURNS = [1, 2]
ALL_UFS = [
    "AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", "MA",
    "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR", "RJ", "RN",
    "RO", "RR", "RS", "SC", "SE", "SP", "TO", "ZZ"
]
TARGET_UFS = ["AC"]
SKIP_EXISTING = True
FORCE_REFRESH = False
STAGING_DIR = Path("data/staging")
EXPORTS_DIR = Path("data/exports")
MANIFEST_PATH = Path("data/exports/_checkpoints/manifest.json")
URL_TEMPLATE = "https://cdn.tse.jus.br/estatistica/sead/eleicoes/eleicoes{election_year}/arqurnatot/bu_imgbu_logjez_rdv_vscmr_{election_year}_{turn}t_{uf}.zip"

# Schema Polars/PyArrow de 15 colunas para fact_urna_logs
FACT_URNA_LOGS_SCHEMA = {
    "election_year": pl.Int32,
    "turn": pl.Int8,
    "uf": pl.Utf8,
    "cd_pleito": pl.Int64,
    "cd_municipio": pl.Int32,
    "nr_zona": pl.Int32,
    "nr_secao": pl.Int32,
    "source_jez_file": pl.Utf8,
    "source_log_member": pl.Utf8,
    "file_seq": pl.Int64,
    "raw_timestamp": pl.Utf8,
    "log_level": pl.Utf8,
    "urna_serial": pl.Utf8,
    "application": pl.Utf8,
    "message_text": pl.Utf8,
}

FACT_URNA_LOGS_ARROW_SCHEMA = pa.schema([
    ("election_year", pa.int32()),
    ("turn", pa.int8()),
    ("uf", pa.string()),
    ("cd_pleito", pa.int64()),
    ("cd_municipio", pa.int32()),
    ("nr_zona", pa.int32()),
    ("nr_secao", pa.int32()),
    ("source_jez_file", pa.string()),
    ("source_log_member", pa.string()),
    ("file_seq", pa.int64()),
    ("raw_timestamp", pa.string()),
    ("log_level", pa.string()),
    ("urna_serial", pa.string()),
    ("application", pa.string()),
    ("message_text", pa.string()),
])